# NB02 — Boolean Search (Method 1 — Gold Standard)
**MSK | Goel Lab** · LLM Systematic Review · Method Comparison

**Purpose**: Execute Boolean search strings across PubMed (and import RIS exports from Embase/Scopus/WoS). Deduplicate results. Run LLM-assisted title/abstract screening. Output the Method 1 retrieved set for evaluation.

**Outputs**
- `data/processed/boolean_search_raw.csv` — all retrieved records pre-dedup
- `data/processed/boolean_search_deduped.csv` — deduplicated corpus
- `data/processed/screening_results_boolean.csv` — screening decisions
- `experiments/<run_id>.json` — run record

**Search strings**: `search/boolean/pubmed_string.txt`, `search/boolean/embase_string.txt`

In [ ]:
import sys
sys.path.insert(0, '..')

from src.utils.io import load_env, data_dir, generate_run_id, log_experiment
load_env()

from pathlib import Path
import pandas as pd
import time

## 1. PubMed search

In [ ]:
from src.methods.boolean_search.fetcher import configure_entrez, search_pubmed, fetch_pubmed_records
import os

configure_entrez(email='your@email.com', api_key=os.getenv('PUBMED_API_KEY'))

pubmed_string = Path('../search/boolean/pubmed_string.txt').read_text()
# Strip comment lines
query = '\n'.join(l for l in pubmed_string.splitlines() if not l.startswith('#'))

t0 = time.perf_counter()
pmids = search_pubmed(query, max_results=10_000)
records = fetch_pubmed_records(pmids)
elapsed = round(time.perf_counter() - t0, 1)
print(f'PubMed: {len(records)} records in {elapsed}s')

## 2. Load RIS exports (Embase / Scopus / WoS)

In [ ]:
from src.methods.boolean_search.fetcher import load_ris

ris_files = list(Path('../search/results').rglob('*.ris'))
print(f'Found {len(ris_files)} RIS files')

import pandas as pd
all_frames = [pd.DataFrame(records)]
for f in ris_files:
    df_ris = load_ris(f)
    all_frames.append(df_ris)

raw_df = pd.concat(all_frames, ignore_index=True)
print(f'Total raw records across all databases: {len(raw_df)}')
raw_df.to_csv(data_dir() / 'processed' / 'boolean_search_raw.csv', index=False)

## 3. Deduplicate

In [ ]:
from src.methods.boolean_search.deduplicator import full_dedup_pipeline

deduped_df = full_dedup_pipeline(raw_df)
deduped_df.to_csv(data_dir() / 'processed' / 'boolean_search_deduped.csv', index=False)
print(f'After deduplication: {len(deduped_df)} records')

## 4. LLM-assisted title/abstract screening

In [ ]:
from src.extraction.extractor import FeatureExtractor
from openai import OpenAI
import json, os

screening_prompt = Path('../prompts/library/screening_v1.txt').read_text()
client = OpenAI()

decisions = []
for _, row in deduped_df.iterrows():
    prompt = screening_prompt.format(title=row.get('title',''), abstract=row.get('abstract',''))
    resp = client.chat.completions.create(
        model='gpt-4o', temperature=0.0,
        response_format={'type': 'json_object'},
        messages=[{'role': 'user', 'content': prompt}]
    )
    d = json.loads(resp.choices[0].message.content)
    d['doc_id'] = row.get('doc_id', '')
    decisions.append(d)

screening_df = pd.DataFrame(decisions)
screening_df.to_csv(data_dir() / 'processed' / 'screening_results_boolean.csv', index=False)
print(screening_df['decision'].value_counts())

## 5. Log run record

In [ ]:
run_id = generate_run_id('boolean')
included = screening_df[screening_df['decision'] == 'include']['doc_id'].tolist()
log_experiment({
    'run_id': run_id,
    'method': 'boolean',
    'n_raw': len(raw_df),
    'n_deduped': len(deduped_df),
    'n_included': len(included),
    'retrieved_doc_ids': included,
    'time_seconds': elapsed,
})
print(f'Run {run_id} logged. Included: {len(included)} articles.')